# GEDI L4A vs. ESA CCI Biomass — Iguazú y Pereyra Iraola
**Pregunta:** ¿otro producto de biomasa independiente de GEDI L4A (ESA CCI Biomass, 100 m, con su incertidumbre) da valores parecidos en el Parque Nacional Iguazú y en la Reserva Pereyra Iraola?

Esto **no valida** ninguno de los dos productos: ambos son modelos satelitales (CCI combina radar y otros sensores; L4A se calibra con parcelas globales y lidar). Sirve para ver si coinciden o difieren, y cuánto. Sin parcelas de campo, toda conclusión es una estimación sobre estimación.

- **CCI Biomass:** `ESA/CCI/Above_Ground_Biomass/V6_0` (bandas `agb` y `agb_sd`, Mg/ha, 100 m; años 2007, 2010, 2015–2022 según el catálogo; **verificar** la versión disponible).
- **GEDI L4A:** `LARSE/GEDI/GEDI04_A_002_MONTHLY`, `agbd` (Mg/ha).
- Comparación por años coincidentes (2019–2022) y por celdas de 100 m con al menos una huella.

In [ ]:
!pip install geemap pandas matplotlib scipy --quiet

## Módulo 0 — Parámetros

In [ ]:
import ee, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats

GEE_PROJECT = 'applied-oxygen-459415-e2'

AOIS = {
    'Iguazú': json.loads('{"type":"Polygon","coordinates":[[[-54.5510497,-25.6276753],[-54.548644,-25.6258187],[-54.5475977,-25.625007],[-54.5463984,-25.6240653],[-54.5408699,-25.6198225],[-54.5386943,-25.6181777],[-54.5376745,-25.6173766],[-54.5371423,-25.6188593],[-54.5366911,-25.619705],[-54.5363237,-25.6203936],[-54.5351092,-25.6219327],[-54.5337952,-25.6229406],[-54.5316523,-25.6245843],[-54.527599,-25.6280658],[-54.5264199,-25.6286655],[-54.5249318,-25.6289605],[-54.5233901,-25.6287255],[-54.5223601,-25.6282795],[-54.5211778,-25.6273935],[-54.5200663,-25.626074],[-54.5196801,-25.625837],[-54.5186758,-25.6252208],[-54.5176641,-25.6248145],[-54.5162608,-25.624325],[-54.5148907,-25.6233867],[-54.5140985,-25.6224123],[-54.5129778,-25.6210339],[-54.5126194,-25.6205029],[-54.5118222,-25.6190818],[-54.5117441,-25.6190137],[-54.511186,-25.6185265],[-54.5103953,-25.6181763],[-54.5093761,-25.6179886],[-54.5078762,-25.6178512],[-54.5070415,-25.6177158],[-54.5059611,-25.6173366],[-54.5053248,-25.61673],[-54.5039902,-25.6154211],[-54.5030482,-25.6148329],[-54.5026978,-25.6147335],[-54.5020247,-25.6145426],[-54.5009003,-25.6144865],[-54.4999604,-25.614622],[-54.4996584,-25.6147443],[-54.4989849,-25.615017],[-54.4984005,-25.6152537],[-54.495965,-25.6165762],[-54.4946452,-25.6171912],[-54.4927034,-25.618096],[-54.4917657,-25.6187983],[-54.4894902,-25.6211598],[-54.4872618,-25.6228604],[-54.4857544,-25.6244508],[-54.4847963,-25.6258583],[-54.4838886,-25.6267511],[-54.4831119,-25.6273064],[-54.4820873,-25.6278713],[-54.4810369,-25.6283173],[-54.4791454,-25.6291714],[-54.477859,-25.6300207],[-54.477129,-25.6306419],[-54.476313,-25.6313363],[-54.4748614,-25.6325437],[-54.4724324,-25.6345641],[-54.471485,-25.6352258],[-54.4704282,-25.6358309],[-54.4700677,-25.6360373],[-54.4691043,-25.6366776],[-54.4683157,-25.6376652],[-54.468164,-25.6378884],[-54.4670422,-25.6395388],[-54.4662021,-25.640595],[-54.4656679,-25.6409529],[-54.4653857,-25.6413543],[-54.465199,-25.6419965],[-54.4648943,-25.6425459],[-54.4644179,-25.6428573],[-54.4635897,-25.6432316],[-54.4621542,-25.6445103],[-54.4605459,-25.6460355],[-54.4602232,-25.6464379],[-54.4592542,-25.6476459],[-54.4591546,-25.647786],[-54.4589529,-25.6480696],[-54.4586946,-25.6484328],[-54.458203,-25.649124],[-54.4574528,-25.6501789],[-54.4554874,-25.6524032],[-54.4551976,-25.6527312],[-54.4538611,-25.6542475],[-54.4526104,-25.6556663],[-54.4516598,-25.6567436],[-54.451023,-25.6574652],[-54.4494609,-25.6598607],[-54.4475991,-25.6623813],[-54.4461257,-25.664376],[-54.4456993,-25.6649532],[-54.4451554,-25.6662713],[-54.4442155,-25.6680496],[-54.4441661,-25.6681975],[-54.4437832,-25.6693444],[-54.443536,-25.6707822],[-54.4435482,-25.671936],[-54.4437499,-25.6725374],[-54.4437274,-25.6734106],[-54.4435289,-25.6744104],[-54.4431872,-25.6754894],[-54.4424788,-25.6784173],[-54.4422509,-25.6802819],[-54.442173,-25.6809191],[-54.4417905,-25.6827585],[-54.4416323,-25.6835194],[-54.4416237,-25.6835607],[-54.4415387,-25.6839696],[-54.4410773,-25.6855378],[-54.4402761,-25.6880054],[-54.4387606,-25.6909829],[-54.4378274,-25.6929456],[-54.4366156,-25.695194],[-54.4363361,-25.6955939],[-54.4358921,-25.6961776],[-54.4356809,-25.6964361],[-54.4355058,-25.6965902],[-54.4347913,-25.697096],[-54.4346336,-25.6971694],[-54.4340414,-25.697445],[-54.4328837,-25.6975601],[-54.4318394,-25.6972834],[-54.4304998,-25.6967596],[-54.4294244,-25.6960173],[-54.4285235,-25.6953955],[-54.4261879,-25.6935953],[-54.4251214,-25.6924032],[-54.4245346,-25.6911009],[-54.4239777,-25.6892649],[-54.4237846,-25.6873883],[-54.4237424,-25.6859561],[-54.4237042,-25.6846607],[-54.4242172,-25.6820785],[-54.4254376,-25.6763612],[-54.4261872,-25.6728498],[-54.4261918,-25.6722372],[-54.4262159,-25.669019],[-54.4262175,-25.6687989],[-54.4261383,-25.6681797],[-54.4254879,-25.6630935],[-54.4235996,-25.6595347],[-54.423008,-25.6589368],[-54.4204448,-25.6563463],[-54.4179348,-25.6538094],[-54.4149975,-25.651569],[-54.4134716,-25.6504051],[-54.4120384,-25.6495543],[-54.4064335,-25.6462269],[-54.4007687,-25.6439056],[-54.3975071,-25.6408105],[-54.3944494,-25.6343879],[-54.3932156,-25.6322986],[-54.3908123,-25.6287389],[-54.3898559,-25.6244633],[-54.3894911,-25.6221222],[-54.3900173,-25.618154],[-54.3901992,-25.6167822],[-54.3910146,-25.6121965],[-54.3916706,-25.608907],[-54.3916084,-25.6087716],[-54.3909042,-25.6072381],[-54.3891365,-25.6033886],[-54.3887738,-25.6025988],[-54.3839459,-25.5974707],[-54.3820216,-25.5967379],[-54.3744043,-25.593837],[-54.3714531,-25.5941873],[-54.3689352,-25.5956181],[-54.3676677,-25.5969105],[-54.3645465,-25.6000929],[-54.3605018,-25.6040792],[-54.3593807,-25.6049645],[-54.3582595,-25.6058498],[-54.3564677,-25.606527],[-54.3543434,-25.6071753],[-54.3541392,-25.607171],[-54.3498012,-25.6070805],[-54.3483138,-25.6070495],[-54.3482718,-25.6070103],[-54.3456908,-25.6046008],[-54.3432865,-25.6023561],[-54.3430734,-25.6019467],[-54.3351974,-25.586815],[-54.3343023,-25.5850952],[-54.3331206,-25.5812078],[-54.3324464,-25.5789896],[-54.33074,-25.5733756],[-54.3247748,-25.5685753],[-54.3191958,-25.566059],[-54.3145113,-25.5645941],[-54.295344,-25.5586],[-54.2940195,-25.5579898],[-54.2881729,-25.5539346],[-54.2839886,-25.5543217],[-54.28,-25.556],[-54.2771145,-25.5605833],[-54.263,-25.583],[-54.255077,-25.5961888],[-54.25,-25.597],[-54.2425622,-25.597],[-54.24,-25.597],[-54.235,-25.593],[-54.2373763,-25.5699693],[-54.2370897,-25.5684534],[-54.2366466,-25.5669484],[-54.2362389,-25.5656515],[-54.2351231,-25.564432],[-54.2335567,-25.563077],[-54.231905,-25.5622956],[-54.23,-25.562],[-54.2268956,-25.5631554],[-54.2243636,-25.5638329],[-54.2216277,-25.5653525],[-54.2200077,-25.5665623],[-54.2170036,-25.5693302],[-54.214139,-25.5721949],[-54.2099869,-25.5756498],[-54.2054594,-25.5786982],[-54.2038179,-25.5796563],[-54.1999233,-25.581224],[-54.1930246,-25.5840207],[-54.1897416,-25.5851045],[-54.186759,-25.5857431],[-54.1849887,-25.5858302],[-54.1811478,-25.5855109],[-54.1787553,-25.5848722],[-54.1777897,-25.5844464],[-54.1768992,-25.5840691],[-54.175719,-25.583082],[-54.1740346,-25.5813208],[-54.1732299,-25.5798305],[-54.1729081,-25.5786208],[-54.1728115,-25.5773241],[-54.1729832,-25.5764434],[-54.1735411,-25.5748853],[-54.1745925,-25.572853],[-54.1764701,-25.5706464],[-54.1814161,-25.5662525],[-54.1831326,-25.5653235],[-54.1862011,-25.5635426],[-54.1941512,-25.5618682],[-54.1959107,-25.5615972],[-54.1999018,-25.5605712],[-54.2019081,-25.5597582],[-54.2039037,-25.5586161],[-54.2050088,-25.557774],[-54.20753,-25.5542508],[-54.2082059,-25.5522278],[-54.2086673,-25.5489561],[-54.2084849,-25.5468556],[-54.207648,-25.5438257],[-54.2068327,-25.5421123],[-54.206,-25.541],[-54.2054165,-25.5401472],[-54.204483,-25.5392469],[-54.2026484,-25.5379304],[-54.2005348,-25.5366332],[-54.1985285,-25.535636],[-54.195739,-25.534668],[-54.1937327,-25.5345518],[-54.1905785,-25.5346486],[-54.1886687,-25.5350455],[-54.1860294,-25.5360426],[-54.1832721,-25.5370785],[-54.1823816,-25.5373786],[-54.1797638,-25.5385596],[-54.1766417,-25.5402537],[-54.1729725,-25.5417251],[-54.1697645,-25.5425963],[-54.1684771,-25.5427996],[-54.1665459,-25.5428868],[-54.1657734,-25.542848],[-54.1642891,-25.5423359],[-54.1640873,-25.5516862],[-54.1633854,-25.584607],[-54.1633464,-25.5854897],[-54.1632683,-25.5896117],[-54.1627014,-25.6153037],[-54.1626912,-25.6166869],[-54.162287,-25.6339453],[-54.1616341,-25.6625931],[-54.1617689,-25.6657826],[-54.161778,-25.6688509],[-54.161962,-25.6805934],[-54.1621114,-25.6901935],[-54.1625758,-25.6980392],[-54.1659409,-25.7613087],[-54.3368356,-25.7559896],[-54.3661128,-25.754991],[-54.3720359,-25.754789],[-54.3923918,-25.7540833],[-54.3918701,-25.7526861],[-54.3917614,-25.7504148],[-54.3920203,-25.7486661],[-54.3923422,-25.7478931],[-54.3939071,-25.7447713],[-54.3953677,-25.743467],[-54.3972989,-25.7416889],[-54.3996593,-25.7403552],[-54.4016334,-25.7389056],[-54.4024044,-25.7376392],[-54.4036504,-25.7345179],[-54.4037791,-25.7335514],[-54.4036289,-25.7326236],[-54.4033929,-25.7314831],[-54.4030588,-25.7303132],[-54.40335,-25.7297241],[-54.4035431,-25.7288349],[-54.4038635,-25.727781],[-54.4036933,-25.7271338],[-54.4035216,-25.7263219],[-54.4036504,-25.7257613],[-54.4042083,-25.7252007],[-54.4048735,-25.7244661],[-54.4054635,-25.7236252],[-54.4057318,-25.7232096],[-54.4055742,-25.7227701],[-54.4065533,-25.7228411],[-54.4086625,-25.7230139],[-54.4088657,-25.7230602],[-54.4090315,-25.7231233],[-54.4105641,-25.7239434],[-54.4135552,-25.7256454],[-54.4160036,-25.7270225],[-54.4162971,-25.7271337],[-54.4164802,-25.727182],[-54.4166726,-25.7271966],[-54.416857,-25.7271808],[-54.417032,-25.7271482],[-54.4174773,-25.7269936],[-54.4181103,-25.726723],[-54.4189632,-25.7263847],[-54.420138,-25.7258966],[-54.4210042,-25.725534],[-54.4317976,-25.7281847],[-54.4457772,-25.7329883],[-54.4463539,-25.7354843],[-54.4473624,-25.7389828],[-54.4482708,-25.7420772],[-54.4893969,-25.7246614],[-54.4880488,-25.7178571],[-54.4883814,-25.7155034],[-54.4892398,-25.7130675],[-54.4902268,-25.7082341],[-54.4968787,-25.699263],[-54.5000544,-25.6934623],[-54.5012131,-25.6890536],[-54.5058372,-25.6848381],[-54.5064505,-25.6830726],[-54.5072213,-25.6809318],[-54.5077514,-25.680097],[-54.5080223,-25.6799665],[-54.508667,-25.6794138],[-54.5094529,-25.678998],[-54.5174351,-25.6756718],[-54.5204392,-25.6697152],[-54.5234433,-25.6666981],[-54.527649,-25.6636036],[-54.5305136,-25.6622981],[-54.530814,-25.6605961],[-54.5313397,-25.659658],[-54.5332038,-25.6598974],[-54.5337671,-25.6596653],[-54.5344618,-25.6592277],[-54.5358994,-25.6577674],[-54.5363903,-25.6568946],[-54.5368141,-25.6560339],[-54.5385173,-25.6535121],[-54.538461,-25.6519623],[-54.538528,-25.6508282],[-54.5389974,-25.6497185],[-54.5393407,-25.6486739],[-54.5398477,-25.6479268],[-54.5399845,-25.6475714],[-54.5403385,-25.6465075],[-54.5406524,-25.6452864],[-54.5415187,-25.6445392],[-54.5429054,-25.6439274],[-54.5440587,-25.6422397],[-54.5448339,-25.6417246],[-54.5457244,-25.6406172],[-54.5467195,-25.6397007],[-54.5467464,-25.6385787],[-54.5483637,-25.6374059],[-54.5486721,-25.6370456],[-54.548766,-25.6362887],[-54.548597,-25.6352562],[-54.5492555,-25.6348439],[-54.549513,-25.6347749],[-54.5497611,-25.6346903],[-54.5500012,-25.6346202],[-54.5503137,-25.6343856],[-54.5507723,-25.634255],[-54.5510754,-25.6341414],[-54.551577,-25.6340664],[-54.5520812,-25.6340543],[-54.5528752,-25.6341704],[-54.5533818,-25.6342234],[-54.5537536,-25.6342623],[-54.5539843,-25.6342828],[-54.5542324,-25.6342272],[-54.5543759,-25.6342043],[-54.5545448,-25.6340918],[-54.5546199,-25.6340144],[-54.5547742,-25.6338633],[-54.5549727,-25.6337206],[-54.555155,-25.6336602],[-54.5553374,-25.6335199],[-54.5555037,-25.633428],[-54.5557022,-25.6333071],[-54.5559329,-25.63322],[-54.5561421,-25.6331886],[-54.5562413,-25.6331499],[-54.5563003,-25.6330169],[-54.5563191,-25.6328742],[-54.5563754,-25.6327412],[-54.5564666,-25.6326917],[-54.5566142,-25.6326832],[-54.5568207,-25.6326542],[-54.5571908,-25.6324135],[-54.5510497,-25.6276753]]]}'),            # OSM (ODbL): Parque Nacional + Reserva Nacional
    'Pereyra Iraola': json.loads('{"type":"Polygon","coordinates":[[[-58.1607722,-34.8315994],[-58.1604567,-34.831347],[-58.1559184,-34.8276218],[-58.1507364,-34.8234119],[-58.1436065,-34.8173],[-58.1434187,-34.8172481],[-58.1428533,-34.8168701],[-58.1421913,-34.8163106],[-58.141883,-34.8160628],[-58.1415552,-34.8158709],[-58.1412436,-34.8155885],[-58.140945,-34.8153594],[-58.1407243,-34.8151195],[-58.1406107,-34.814949],[-58.1404193,-34.8147705],[-58.1402408,-34.8146932],[-58.1399973,-34.8146799],[-58.1398383,-34.8146559],[-58.1396598,-34.8145307],[-58.1394878,-34.8143255],[-58.1393872,-34.8142855],[-58.1392541,-34.8142509],[-58.1390524,-34.8141604],[-58.1387868,-34.8139331],[-58.1384874,-34.8137522],[-58.138137,-34.8136919],[-58.1378262,-34.8136362],[-58.1376059,-34.8134924],[-58.1375154,-34.8133532],[-58.137538,-34.8131769],[-58.1375098,-34.812996],[-58.137329,-34.8128011],[-58.1372216,-34.8126063],[-58.1367131,-34.812031],[-58.1364927,-34.8118964],[-58.136244,-34.8117851],[-58.1359672,-34.8116181],[-58.1357298,-34.8114093],[-58.1353625,-34.8111681],[-58.1350744,-34.8110567],[-58.1348088,-34.8110474],[-58.1345206,-34.8110335],[-58.1343793,-34.8109314],[-58.1343398,-34.8107412],[-58.1342381,-34.8106856],[-58.1340177,-34.8105557],[-58.1338199,-34.8105371],[-58.133656,-34.8105649],[-58.1335148,-34.8106716],[-58.1334074,-34.8106948],[-58.1332661,-34.8106113],[-58.1329102,-34.8104536],[-58.1324355,-34.8104072],[-58.132136,-34.8103933],[-58.131684,-34.8104443],[-58.1313788,-34.8105371],[-58.1311867,-34.8105649],[-58.1309776,-34.8105],[-58.1307968,-34.8103283],[-58.1305651,-34.8101335],[-58.1303504,-34.809985],[-58.1300961,-34.8097206],[-58.1299662,-34.8094329],[-58.1299153,-34.8092241],[-58.1301639,-34.8090246],[-58.1302995,-34.808839],[-58.1302939,-34.8086906],[-58.1300114,-34.808389],[-58.1295028,-34.8077766],[-58.1292259,-34.8073126],[-58.1287852,-34.8066352],[-58.1281749,-34.8056516],[-58.127898,-34.8052155],[-58.1275251,-34.8046819],[-58.1271578,-34.8043061],[-58.1262085,-34.8034384],[-58.1252139,-34.8025337],[-58.1248805,-34.8022599],[-58.1247412,-34.8021299],[-58.1245581,-34.801959],[-58.1240888,-34.8015493],[-58.1226296,-34.8002543],[-58.1217713,-34.799457],[-58.1197087,-34.7976443],[-58.1062574,-34.7856571],[-58.0853818,-34.7668314],[-58.0841319,-34.7657076],[-58.083423,-34.764926],[-58.08316,-34.7650109],[-58.0818809,-34.7656298],[-58.0802715,-34.7664822],[-58.0791771,-34.7671434],[-58.0775893,-34.7681393],[-58.0687809,-34.7719907],[-58.0653584,-34.7729866],[-58.05413,-34.7765399],[-58.05403,-34.7769601],[-58.0538941,-34.7770509],[-58.0535619,-34.7771158],[-58.0534708,-34.7768377],[-58.0531467,-34.7765734],[-58.0530795,-34.7764467],[-58.0431279,-34.7772916],[-58.037903,-34.7773797],[-58.0347916,-34.7775912],[-58.0302104,-34.7780054],[-58.0259403,-34.7784196],[-58.021901,-34.7792067],[-58.0209312,-34.7794966],[-58.0183129,-34.7805375],[-58.0171797,-34.7799102],[-58.0145699,-34.7806385],[-58.0131022,-34.7812108],[-58.0120613,-34.7818022],[-58.011405,-34.7821368],[-58.0111661,-34.7822828],[-58.0110303,-34.7823727],[-58.0108949,-34.7824036],[-58.0100706,-34.7829188],[-58.0096319,-34.7828506],[-58.009529,-34.7828605],[-58.0095263,-34.7828618],[-58.0094726,-34.7828873],[-58.0094165,-34.7829139],[-58.0093329,-34.783042],[-58.0093507,-34.7832435],[-58.0085307,-34.7834796],[-58.0073558,-34.7838082],[-58.0068791,-34.7840631],[-58.0068581,-34.7841497],[-58.005408,-34.7848325],[-58.0049479,-34.7849683],[-58.0048689,-34.7849185],[-58.0044056,-34.7850853],[-58.0030532,-34.7858616],[-58.0030578,-34.7859449],[-58.0025669,-34.7863214],[-58.0022817,-34.7864641],[-58.0021452,-34.7864604],[-58.001765,-34.7866756],[-58.0001005,-34.7881085],[-57.9994999,-34.7884897],[-57.9985948,-34.7896046],[-57.9983787,-34.7899813],[-57.9984215,-34.7900701],[-57.9981878,-34.7904034],[-57.9977902,-34.7906336],[-57.9969603,-34.7911139],[-57.9968074,-34.7911051],[-57.9967457,-34.7910985],[-57.9966679,-34.7911293],[-57.9963487,-34.7914421],[-57.9961838,-34.7916139],[-57.9961099,-34.7917333],[-57.9962437,-34.7918651],[-57.9975,-34.7930071],[-57.9987992,-34.7941725],[-57.9989434,-34.7943041],[-58.005169,-34.7999882],[-58.0053289,-34.8000978],[-58.0056788,-34.7997628],[-58.005744,-34.7997549],[-58.005804,-34.7997945],[-58.0077989,-34.8016153],[-58.0503106,-34.8393519],[-58.0559364,-34.8444066],[-58.0568718,-34.8452653],[-58.0582643,-34.8465432],[-58.0663437,-34.8537898],[-58.0709206,-34.8507106],[-58.0706968,-34.8504095],[-58.0707866,-34.8502178],[-58.0746697,-34.8483312],[-58.0745561,-34.8481689],[-58.0770406,-34.8464086],[-58.0776631,-34.8459608],[-58.0864901,-34.8399418],[-58.0876379,-34.839136],[-58.0879671,-34.8388901],[-58.0884804,-34.8387354],[-58.0894812,-34.8393824],[-58.0989874,-34.8473138],[-58.1105029,-34.8562688],[-58.1216508,-34.8658583],[-58.1369662,-34.8782651],[-58.1478158,-34.8872667],[-58.1535944,-34.882937],[-58.1651427,-34.8739723],[-58.1719279,-34.8688635],[-58.1804362,-34.8626759],[-58.1844186,-34.8521907],[-58.1845008,-34.8519512],[-58.1841908,-34.8516437],[-58.1822363,-34.849943],[-58.1840942,-34.8484565],[-58.1841453,-34.8484435],[-58.1856353,-34.8488095],[-58.188045,-34.842184],[-58.188045,-34.8419463],[-58.188008,-34.8416352],[-58.1879464,-34.841481],[-58.1884209,-34.8412686],[-58.1884086,-34.8412104],[-58.1883522,-34.8411436],[-58.1882555,-34.8410677],[-58.1882098,-34.841042],[-58.1881547,-34.8410163],[-58.1881042,-34.8410002],[-58.1880491,-34.8409879],[-58.1879101,-34.8409746],[-58.1878086,-34.8409822],[-58.1876954,-34.8409955],[-58.1872804,-34.8410613],[-58.1868084,-34.8411107],[-58.1863317,-34.8411294],[-58.1744075,-34.8397672],[-58.1733697,-34.8396713],[-58.1727009,-34.8396291],[-58.1719251,-34.8396425],[-58.1711586,-34.8397229],[-58.1701738,-34.8399206],[-58.1693256,-34.8404632],[-58.163532,-34.84233],[-58.1594122,-34.8434218],[-58.1573093,-34.8445137],[-58.1555498,-34.844725],[-58.1495651,-34.840309],[-58.1478578,-34.8388984],[-58.1513549,-34.8361717],[-58.151991,-34.8357045],[-58.1495953,-34.8323969],[-58.1552071,-34.8292845],[-58.1602907,-34.8317883],[-58.1607722,-34.8315994]]]}'),    # OSM (ODbL), relación 10968584
}
ANIOS = [2019, 2020, 2021, 2022]       # años con CCI y GEDI en común
CCI = 'ESA/CCI/Above_Ground_Biomass/V6_0'
GEDI_L4A = 'LARSE/GEDI/GEDI04_A_002_MONTHLY'
SENS_MIN = 0.95

In [ ]:
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    print('Iniciando autenticación de Earth Engine...')
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

cci = ee.ImageCollection(CCI)
print('Bandas CCI:', cci.first().bandNames().getInfo())
print('Años CCI disponibles:', sorted(pd.to_datetime(cci.aggregate_array('system:time_start').getInfo(), unit='ms').year.tolist()))

## Módulo 1 — Medias por área (CCI por año)
Media de `agb` sobre el polígono, y la media de su incertidumbre `agb_sd`.

In [ ]:
def cci_anio(y):
    col = cci.filterDate(f'{y}-01-01', f'{y}-12-31')
    assert col.size().getInfo() > 0, f'CCI sin imagen para {y}'
    return col.first()

filas = []
for nombre, g in AOIS.items():
    aoi = ee.Geometry(g)
    for y in ANIOS:
        r = cci_anio(y).select(['agb', 'agb_sd']).reduceRegion(ee.Reducer.mean(), aoi, 100, maxPixels=1e10, tileScale=4).getInfo()
        filas.append({'sitio': nombre, 'anio': y, 'CCI_agb_media': r['agb'], 'CCI_agb_sd_media': r['agb_sd']})
medias_cci = pd.DataFrame(filas).round(1)
print(medias_cci.to_string(index=False))

## Módulo 2 — Huellas GEDI L4A + valor de CCI en cada huella (extracción liviana)
Para que corra rápido **no se promedia GEDI a la grilla de CCI** (`reduceResolution`, lo que causaba el timeout). Se extraen las huellas de calidad en su resolución nativa (25 m) junto con el valor de CCI de la celda de 100 m donde cae cada una, en un solo `reduceRegion` por tesela. Después, las celdas de 100 m y los bloques de 1 km se arman en pandas. Los resultados se guardan por sitio y año: si se corta, al volver a correr retoma desde lo ya guardado.

In [ ]:
import os
SENS_MIN = 0.95
NT = 2          # teselas por lado (2x2)


def teselas(aoi_geom, nx=NT, ny=NT):
    b = aoi_geom.bounds().coordinates().getInfo()[0]
    xs = [p[0] for p in b]
    ys = [p[1] for p in b]
    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    return [ee.Geometry.Rectangle([x0 + i * (x1 - x0) / nx, y0 + j * (y1 - y0) / ny, x0 + (i + 1) * (x1 - x0) / nx, y0 + (j + 1) * (y1 - y0) / ny]).intersection(aoi_geom, 1)
            for i in range(nx) for j in range(ny)]


def extraer(nombre, y):
    archivo = f'cache_{nombre[:3]}_{y}.csv'
    if os.path.exists(archivo):
        return pd.read_csv(archivo)
    aoi = ee.Geometry(AOIS[nombre])
    col = ee.ImageCollection(GEDI_L4A).filterDate(f'{y}-01-01', f'{y}-12-31').filterBounds(aoi)

    def calidad(im):
        m = im.select('l4_quality_flag').eq(1).And(im.select('degrade_flag').eq(0)).And(im.select('sensitivity').gte(SENS_MIN))
        return im.select(['agbd']).updateMask(m)

    g = col.map(calidad).mosaic().rename('gedi_agbd')
    ci = cci_anio(y).select(['agb', 'agb_sd'])
    st = g.addBands(ci).addBands(ee.Image.pixelLonLat())
    st = st.updateMask(st.select(['gedi_agbd', 'agb', 'agb_sd']).mask().reduce(ee.Reducer.min()))
    cols = ['gedi_agbd', 'agb', 'agb_sd', 'longitude', 'latitude']
    filas = []
    for t in teselas(aoi):
        r = st.reduceRegion(ee.Reducer.toList(), t, 25, maxPixels=1e10, tileScale=4).getInfo()
        if r.get('agb'):
            n = min(len(r[k]) for k in cols)
            filas.append(pd.DataFrame({k: r[k][:n] for k in cols}))
    d = pd.concat(filas, ignore_index=True) if filas else pd.DataFrame(columns=cols)
    d['sitio'] = nombre
    d['anio'] = y
    d.to_csv(archivo, index=False)
    return d


partes = []
for n_ in AOIS:
    for y in ANIOS:
        d = extraer(n_, y)
        print(n_, y, len(d), 'huellas', flush=True)
        partes.append(d)
huellas = pd.concat(partes, ignore_index=True).rename(columns={'agb': 'cci_agb', 'agb_sd': 'cci_agb_sd'})

# celdas de 100 m (en pandas): se agrupan las huellas que caen en la misma celda
lat0 = huellas.latitude.mean()
huellas['cx'] = np.floor(huellas.longitude * 111.32e3 * np.cos(np.radians(lat0)) / 100)
huellas['cy'] = np.floor(huellas.latitude * 110.57e3 / 100)
pares = huellas.groupby(['sitio', 'anio', 'cx', 'cy']).agg(n_huellas=('gedi_agbd', 'size'), gedi_agbd=('gedi_agbd', 'mean'), cci_agb=('cci_agb', 'mean'),
                                                          cci_agb_sd=('cci_agb_sd', 'mean'), longitude=('longitude', 'mean'), latitude=('latitude', 'mean')).reset_index()
print(pares.groupby(['sitio', 'anio']).size().unstack())

## Módulo 3 — Resultados

In [ ]:
def resumen(d):
    r = stats.pearsonr(d.gedi_agbd, d.cci_agb)[0] if len(d) > 2 else np.nan
    return pd.Series({'celdas_100m': len(d), 'GEDI_L4A_media': d.gedi_agbd.mean(), 'CCI_media': d.cci_agb.mean(),
                      'CCI/GEDI': d.cci_agb.mean() / d.gedi_agbd.mean(), 'sesgo_CCI-GEDI': (d.cci_agb - d.gedi_agbd).mean(),
                      'r': r, 'CCI_sd_media': d.cci_agb_sd.mean()})


res = pares.groupby('sitio')[list(pares.columns)].apply(resumen).round(2)
print(res.to_string())
print()
print(pares.groupby(['sitio', 'anio'])[list(pares.columns)].apply(resumen).round(2).to_string())

In [ ]:
colores = {'Iguazú': '#eb6834', 'Pereyra Iraola': '#2a78d6'}
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for a, (nombre, d) in zip(ax, pares.groupby('sitio')):
    a.scatter(d.gedi_agbd, d.cci_agb, s=6, alpha=.35, color=colores[nombre])
    lim = max(d.gedi_agbd.quantile(.99), d.cci_agb.quantile(.99))
    a.plot([0, lim], [0, lim], 'k--', lw=1)
    a.set_xlim(0, lim)
    a.set_ylim(0, lim)
    a.set_title(f'{nombre} (n = {len(d)})')
    a.set_xlabel('GEDI L4A, AGBD media por celda (Mg/ha)')
    a.set_ylabel('ESA CCI Biomass, AGB (Mg/ha)')
plt.suptitle('Celdas de 100 m con huellas GEDI, años 2019–2022 · línea: 1:1')
plt.tight_layout()
plt.savefig('cci_vs_gedi.png', dpi=150)
pares.to_csv('pares_cci_gedi.csv', index=False)
res.to_csv('resumen_cci_gedi.csv')
medias_cci.to_csv('medias_cci_por_area.csv', index=False)

## Módulo 4 — Comparación a 1 km
A 100 m, CCI es casi plano y las celdas de GEDI tienen pocas huellas (ruido). Acá se agrupan las celdas de 100 m en bloques de ~1 km (con al menos `MIN_CELDAS` celdas) y se compara la media de cada bloque. Sirve para ver si coinciden en el **patrón espacial** (zonas densas vs. abiertas), más allá del nivel absoluto.

In [ ]:
MIN_CELDAS = 5

def bloques(d, km=1.0):
    d = d.copy()
    lat0 = d.latitude.mean()
    d['bx'] = np.floor(d.longitude * 111.32 * np.cos(np.radians(lat0)) / km)
    d['by'] = np.floor(d.latitude * 110.57 / km)
    g = d.groupby(['sitio', 'bx', 'by']).agg(celdas=('gedi_agbd', 'size'), gedi=('gedi_agbd', 'mean'), cci=('cci_agb', 'mean'), cci_sd=('cci_agb_sd', 'mean')).reset_index()
    return g[g.celdas >= MIN_CELDAS]

b1 = bloques(pares)
filas = []
for nombre, d in b1.groupby('sitio'):
    filas.append({'sitio': nombre, 'bloques_1km': len(d), 'GEDI_media': d.gedi.mean(), 'CCI_media': d.cci.mean(), 'CCI/GEDI': d.cci.mean() / d.gedi.mean(),
                  'r_pearson': stats.pearsonr(d.gedi, d.cci)[0] if len(d) > 2 else np.nan,
                  'rho_spearman': stats.spearmanr(d.gedi, d.cci)[0] if len(d) > 2 else np.nan})
res1km = pd.DataFrame(filas).round(2)
print(res1km.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
for a, (nombre, d) in zip(ax, b1.groupby('sitio')):
    a.scatter(d.gedi, d.cci, s=d.celdas * 2, alpha=.5, color=colores[nombre])
    lim = max(d.gedi.max(), d.cci.max()) * 1.05
    a.plot([0, lim], [0, lim], 'k--', lw=1)
    a.set_xlim(0, lim)
    a.set_ylim(0, lim)
    a.set_title(f'{nombre} (n = {len(d)} bloques)')
    a.set_xlabel('GEDI L4A, media del bloque (Mg/ha)')
    a.set_ylabel('ESA CCI Biomass, media del bloque (Mg/ha)')
plt.suptitle(f'Bloques de ~1 km con al menos {MIN_CELDAS} celdas · tamaño = nº de celdas · línea: 1:1')
plt.tight_layout()
plt.savefig('cci_vs_gedi_1km.png', dpi=150)
b1.to_csv('bloques_1km_cci_gedi.csv', index=False)
res1km.to_csv('resumen_1km_cci_gedi.csv', index=False)

## Cómo leer el resultado
- **CCI ≈ GEDI (cociente cerca de 1):** dos productos independientes coinciden; la baja biomasa de GEDI en Iguazú no sería un artefacto de L4A.
- **CCI claramente mayor en Iguazú:** apunta a que L4A subestima (o CCI sobreestima) ahí; **sin parcelas no se puede decidir cuál**. Mirar `CCI_sd_media` (la incertidumbre de CCI suele ser del orden del 40 % de la media).
- **Pereyra:** vegetación baja y heterogénea; CCI está pensado para biomasa leñosa, y a 100 m puede no resolver parches chicos.
- Los dos productos tienen sus propios errores; la coincidencia no prueba que ambos sean correctos.